In [ ]:
"""
Aperiodic (1/f) Parameterization of NREM Sleep EEG in IESS
============================================================
 
Purpose

This code lets you test whether the treatment-related changes in theta/alpha/sigma/
gamma/delta power (Pre -> Post1 -> Post2) reflect genuine oscillatory
reorganization, a broadband tilt of the aperiodic slope, or both. It also
gives you an independent, biologically interpretable marker (aperiodic
exponent, linked to cortical E/I balance) to report alongside SWI and PAC.
 
Requirements
------------
pip install --break-system-packages mne specparam pandas scipy statsmodels
 
Expected input
--------------
This script assumes, per subject and per timepoint (Pre / Post1 / Post2), you
have either:
  (a) a continuous MNE-readable raw file (.fif, .edf, .set, .vhdr, ...) plus
      annotations/events marking the 30-s NREM epochs already scored, OR
  (b) a pre-epoched .fif file (mne.Epochs saved with epochs.save) where each
      epoch is a scored 30-s NREM segment.
 
Edit `load_epochs_for_recording()` below to match whichever you actually have
-- that is the only part that is dataset-specific. Everything downstream
(PSD -> specparam fit -> aggregation -> stats) is generic.
 
Directory / manifest convention assumed here (adjust as needed):
    manifest.csv with columns: subject, timepoint, filepath
    e.g.
        subject,timepoint,filepath
        S01,Pre,/data/S01_pre_epo.fif
        S01,Post1,/data/S01_post1_epo.fif
        S01,Post2,/data/S01_post2_epo.fif
        S02,Pre,/data/S02_pre_epo.fif
        ...
"""

In [2]:
import warnings
import numpy as np
import pandas as pd
import mne
!pip install specparam mne pandas scipy statsmodels
from specparam import SpectralModel
from specparam.bands import Bands
from specparam.data.periodic import get_band_peak_arr
 
mne.set_log_level("WARNING")
warnings.filterwarnings("ignore")

Defaulting to user installation because normal site-packages is not writeable


In [4]:
# ----------------------------------------------------------------------
# 1. CONFIGURATION -- match these to your study
# ----------------------------------------------------------------------
 
MANIFEST_CSV = "manifest.csv"          # subject, timepoint, filepath
OUTPUT_CSV = "aperiodic_results.csv"
 
# Frequency range to fit. Excludes very low frequencies (drift/filter
# artifacts) and anything above 45 Hz (your recording's upper bound / line
# noise region). specparam fits are unreliable right at the recording edges.
FIT_FREQ_RANGE = (0.5, 45.0)
 
# Welch PSD parameters. For 30-s epochs, a 4-s window with 50% overlap gives
# ~0.25 Hz resolution -- fine enough to resolve delta/theta/alpha peaks
# without being noisy.
WELCH_N_FFT = None       # set to sfreq*4 once sfreq is known (done per-file below)
WELCH_N_OVERLAP_FRAC = 0.5
 
# Band definitions matching your 8-band scheme (adjust edges to match your
# actual analysis if they differ slightly)
BANDS = Bands({
    "slow_delta": (0.5, 2),
    "delta": (2, 4),
    "theta": (4, 8),
    "alpha": (8, 13),
    "sigma": (13, 16),
    "beta": (16, 30),
    "gamma": (30, 45),
})
 
# Whether to fit with a "knee" term (recommended when the spectrum isn't a
# clean straight line in log-log space across your full fit range, which is
# common in pediatric EEG with a sigma/spindle hump). Try 'fixed' first and
# inspect residuals/reports; switch to 'knee' if fits look poor at low freq.
APERIODIC_MODE = "fixed"   # or "knee"

In [5]:
import mne
raw = mne.io.read_raw_edf(r"J:\Research\Barb Research\Nie, Duyu\Multimodal Biomarker Epilepsies Study\EDF files\Fully De-Identified\ELE03_N2_Pre.edf", preload=False)
print(raw.info)
print(set(raw.annotations.description))
print(raw.annotations.duration[:20])

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, F7, T3, T5, O1, F3, C3, P3, A1, Fz, Cz, Fp2, F8, T4, T6, ...
 chs: 46 EEG
 custom_ref_applied: False
 highpass: 0.0 Hz
 lowpass: 128.0 Hz
 meas_date: 2024-01-04 11:24:22 UTC
 nchan: 46
 projs: []
 sfreq: 256.0 Hz
 subject_info: <subject_info | his_id: X, sex: 0, last_name: ELE,03>
>
set()
[]


In [ ]:
# ----------------------------------------------------------------------
# 2. DATA LOADING -- the one dataset-specific function
# ----------------------------------------------------------------------
 
def load_epochs_for_recording(filepath):
    """
    Load the scored NREM epochs for one subject/timepoint recording.
 
    Returns
    -------
    epochs : mne.Epochs
        Each epoch = one scored 30-s NREM segment, EEG channels only,
        already artifact-cleaned.
    """
    if filepath.endswith("-epo.fif") or filepath.endswith("_epo.fif"):
        epochs = mne.read_epochs(filepath, preload=True)
    else:
        # Continuous file: read raw, then epoch using existing annotations.
        # Adjust event_id / annotation description to match your scoring
        # convention (e.g. 'NREM', 'N2', 'N3', etc.)
        raw = mne.io.read_raw(filepath, preload=True)
        events, event_id = mne.events_from_annotations(raw)
        nrem_ids = {k: v for k, v in event_id.items() if "NREM" in k.upper()}
        epochs = mne.Epochs(
            raw, events, event_id=nrem_ids,
            tmin=0, tmax=30.0, baseline=None, preload=True,
        )
    epochs.pick("eeg")
    return epochs
 

In [ ]:
# ----------------------------------------------------------------------
# 3. PSD COMPUTATION (per epoch, per channel)
# ----------------------------------------------------------------------
 
def compute_epoch_psds(epochs):
    """
    Welch PSD per epoch, per channel.
 
    Returns
    -------
    freqs : ndarray, shape (n_freqs,)
    psds  : ndarray, shape (n_epochs, n_channels, n_freqs)
    ch_names : list of channel names
    """
    sfreq = epochs.info["sfreq"]
    n_fft = int(sfreq * 4)          # 4-second Welch window
    n_overlap = int(n_fft * WELCH_N_OVERLAP_FRAC)
 
    spectrum = epochs.compute_psd(
        method="welch",
        fmin=0.5, fmax=45.0,
        n_fft=n_fft, n_overlap=n_overlap,
        average="mean",
    )
    psds, freqs = spectrum.get_data(return_freqs=True)  # (n_epochs, n_ch, n_freqs)
    return freqs, psds, epochs.ch_names
 

In [ ]:
# ----------------------------------------------------------------------
# 4. SPECPARAM (FOOOF) FITTING
# ----------------------------------------------------------------------
 
def fit_aperiodic(freqs, psd_1d):
    """
    Fit one power spectrum (already averaged or single epoch/channel).
 
    Returns dict with aperiodic offset/exponent (+ knee if used) and,
    for convenience, per-band peak power extracted from the *periodic*
    component only (i.e. oscillatory power with the 1/f background removed).
    """
    model = SpectralModel(
        aperiodic_mode=APERIODIC_MODE,
        peak_width_limits=(1.0, 8.0),
        max_n_peaks=8,
        min_peak_height=0.05,
        peak_threshold=2.0,
        verbose=False,
    )
    model.fit(freqs, psd_1d, freq_range=FIT_FREQ_RANGE)
 
    out = {}
    ap_params = model.get_params("aperiodic")
    if APERIODIC_MODE == "knee":
        out["offset"], out["knee"], out["exponent"] = ap_params
    else:
        out["offset"], out["exponent"] = ap_params
        out["knee"] = np.nan
 
    metrics = model.results.metrics.results
    out["r_squared"] = metrics.get("gof_rsquared", np.nan)
    out["error"] = metrics.get("error_mae", np.nan)
 
    # Periodic (peak) power per band, aperiodic-corrected
    peak_params = model.get_params("peak")
    for band_name, band_range in BANDS:
        band_peak = get_band_peak_arr(peak_params, band_range, select_highest=True)
        # band_peak is [CF, PW, BW], filled with nan if no peak found in range
        out[f"periodic_{band_name}_cf"] = band_peak[0]
        out[f"periodic_{band_name}_power"] = band_peak[1]
        out[f"periodic_{band_name}_bw"] = band_peak[2]
 
    return out
 

In [ ]:
# ----------------------------------------------------------------------
# 5. MAIN PIPELINE
# ----------------------------------------------------------------------
 
def run_pipeline(manifest_csv=MANIFEST_CSV, output_csv=OUTPUT_CSV):
    manifest = pd.read_csv(manifest_csv)
    rows = []
 
    for _, row in manifest.iterrows():
        subject, timepoint, filepath = row["subject"], row["timepoint"], row["filepath"]
        print(f"Processing {subject} / {timepoint} ...")
 
        epochs = load_epochs_for_recording(filepath)
        freqs, psds, ch_names = compute_epoch_psds(epochs)  # (n_ep, n_ch, n_freq)
 
        # Average PSD across epochs first (recording-level spectrum), then
        # fit once per channel. This matches how band power is typically
        # summarized in your pipeline (per-channel, then averaged across a
        # region/all channels). If you'd rather fit epoch-by-epoch to get a
        # distribution (recommended if you want within-recording variance),
        # see `fit_aperiodic_epochwise()` below instead.
        mean_psd = psds.mean(axis=0)  # (n_ch, n_freq)
 
        for ch_idx, ch_name in enumerate(ch_names):
            try:
                fit_result = fit_aperiodic(freqs, mean_psd[ch_idx])
            except Exception as e:
                print(f"  fit failed for {ch_name}: {e}")
                continue
            fit_result.update({
                "subject": subject,
                "timepoint": timepoint,
                "channel": ch_name,
            })
            rows.append(fit_result)
 
    results_df = pd.DataFrame(rows)
    results_df.to_csv(output_csv, index=False)
    print(f"\nSaved {len(results_df)} rows to {output_csv}")
    return results_df
 
 
def fit_aperiodic_epochwise(epochs, freqs=None, psds=None):
    """
    Alternative: fit every single epoch (not just the recording mean),
    per channel, so you get a distribution of exponent/offset values per
    subject/timepoint suitable for a linear mixed-effects model with
    epoch as the repeated-measures unit (mirrors your SWI/spectral LME
    approach). Returns a long-format DataFrame.
    """
    if freqs is None or psds is None:
        freqs, psds, ch_names = compute_epoch_psds(epochs)
    else:
        ch_names = epochs.ch_names
 
    rows = []
    n_epochs, n_ch, _ = psds.shape
    for ep_idx in range(n_epochs):
        for ch_idx, ch_name in enumerate(ch_names):
            try:
                fit_result = fit_aperiodic(freqs, psds[ep_idx, ch_idx])
            except Exception:
                continue
            fit_result.update({"epoch": ep_idx, "channel": ch_name})
            rows.append(fit_result)
    return pd.DataFrame(rows)
 

In [ ]:
# ----------------------------------------------------------------------
# 6. STATISTICS -- Linear Mixed-Effects across Pre/Post1/Post2
# ----------------------------------------------------------------------
 
def run_lme(results_df, dependent_var="exponent"):
    """
    Mirrors the LME approach in your spectral analysis: timepoint as a
    fixed effect, subject (and optionally channel) as random effects.
    Requires >1 observation per subject/timepoint (e.g. per-channel or
    per-epoch rows) for the random effect to be estimable.
    """
    import statsmodels.formula.api as smf
 
    df = results_df.copy()
    df["timepoint"] = pd.Categorical(df["timepoint"], categories=["Pre", "Post1", "Post2"], ordered=True)
 
    model = smf.mixedlm(
        f"{dependent_var} ~ C(timepoint, Treatment(reference='Pre'))",
        data=df,
        groups=df["subject"],
    )
    result = model.fit()
    print(result.summary())
    return result
 
 
if __name__ == "__main__":
    df = run_pipeline()
    print("\n--- Aperiodic exponent across timepoints (mean +/- SD) ---")
    print(df.groupby("timepoint")["exponent"].agg(["mean", "std", "count"]))
 
    print("\n--- LME: exponent ~ timepoint ---")
    run_lme(df, dependent_var="exponent")
 
    print("\n--- LME: offset ~ timepoint ---")
    run_lme(df, dependent_var="offset")
 